# ============================================================
# STEP 3 — MODEL TRAINING & EVALUATION (TUMC)
# Transductive vs Inductive graph-feature comparison
# ============================================================
# Runs the FULL model suite on BOTH feature sets:
#   GRAPH_MODE="transductive" → features_full_final.csv
#       (graph stats computed over the whole corpus; upper bound)
#   GRAPH_MODE="inductive"    → features_full_final_inductive.csv
#       (graph stats train-only per fold; deployment estimate)
#
# Both write to the same leaderboard with a 'graph_mode' column,
# so the transductive-vs-inductive gap is directly reportable.
#
# 2 tasks × 3 variants × up to 9 models × 2 graph modes.
# Start with the quick-test config at the bottom of CONFIG.
# ============================================================

In [ ]:
import os, time, pickle, warnings
import numpy as np
import pandas as pd
from sklearn.ensemble import (
    HistGradientBoostingClassifier, RandomForestClassifier, ExtraTreesClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import ComplementNB
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    matthews_corrcoef, balanced_accuracy_score, accuracy_score,
    average_precision_score)

warnings.filterwarnings("ignore")
SEED = 42
DATA_DIR = "/content/drive/MyDrive/TUMC_dataset"
OUT_METRICS = os.path.join(DATA_DIR, "step3_all_metrics_final.csv")

# The two feature files produced by Step 2D and Step 2D-inductive
FEATURE_FILES = {
    "transductive": os.path.join(DATA_DIR, "features_full_final.csv"),
    "inductive":    os.path.join(DATA_DIR, "features_full_final_inductive.csv"),
}

HAS_XGB = HAS_LGB = HAS_CAT = HAS_IMB = False
try: import xgboost as xgb; HAS_XGB = True
except ImportError: pass
try: import lightgbm as lgb; HAS_LGB = True
except ImportError: pass
try:
    from imblearn.ensemble import BalancedRandomForestClassifier
    HAS_IMB = True
except ImportError: pass

# ════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════
GRAPH_MODES = ["transductive", "inductive"]   # run both
TASKS    = ["binary", "5class"]
VARIANTS = {
    "A_all":         [],
    "B_no_tr":       ["is_tr_domain"],
    "C_no_tr_https": ["is_tr_domain", "is_https"],
}
# Graph features flagged leaky in Step 2D — always drop for honesty
LEAKY_GRAPH = ["cluster_malicious_ratio", "campaign_token_score"]

# QUICK TEST (uncomment to validate before the full grid):
# GRAPH_MODES = ["inductive"]; TASKS = ["binary"]
# VARIANTS = {"C_no_tr_https": ["is_tr_domain", "is_https"]}

NEEDS_SCALING = {"LogisticRegression"}
NEEDS_MINMAX  = {"ComplementNB"}

def make_models(n_classes):
    is_binary = (n_classes == 2)
    M = {}
    M["HistGB"] = HistGradientBoostingClassifier(
        max_depth=6, learning_rate=0.03, max_iter=500, random_state=SEED)
    M["RandomForest"] = RandomForestClassifier(
        n_estimators=300, max_depth=12, min_samples_split=10, min_samples_leaf=5,
        class_weight="balanced", random_state=SEED, n_jobs=-1)
    #M["ExtraTrees"] = ExtraTreesClassifier( n_estimators=300, max_depth=12, min_samples_split=10, min_samples_leaf=5, max_features="sqrt", bootstrap=True, class_weight="balanced",random_state=SEED, n_jobs=-1)
    M["LogisticRegression"] = LogisticRegression(
        C=1.0, max_iter=1000, class_weight="balanced", random_state=SEED, n_jobs=-1)
    M["ComplementNB"] = ComplementNB(alpha=1.0)
    #if HAS_IMB:M["BalancedRF"] = BalancedRandomForestClassifier(n_estimators=300, random_state=SEED, n_jobs=-1)
    if HAS_XGB:
        if is_binary:
            M["XGBoost"] = xgb.XGBClassifier(
                objective="binary:logistic", eval_metric="logloss",
                max_depth=4, learning_rate=0.03, n_estimators=500,
                subsample=0.8, colsample_bytree=0.8, reg_alpha=1, reg_lambda=2,
                scale_pos_weight=5.5, random_state=SEED, n_jobs=-1)
        else:
            M["XGBoost"] = xgb.XGBClassifier(
                objective="multi:softprob", num_class=n_classes, eval_metric="mlogloss",
                max_depth=4, learning_rate=0.03, n_estimators=500,
                subsample=0.8, colsample_bytree=0.8, reg_alpha=1, reg_lambda=2,
                random_state=SEED, n_jobs=-1)
    if HAS_LGB:
        obj = "binary" if is_binary else "multiclass"
        kw = {} if is_binary else {"num_class": n_classes}
        M["LightGBM"] = lgb.LGBMClassifier(
            objective=obj, num_leaves=31, max_depth=6, learning_rate=0.03,
            n_estimators=500, feature_fraction=0.8, bagging_fraction=0.8,
            bagging_freq=5, reg_alpha=1, reg_lambda=1, class_weight="balanced",
            random_state=SEED, n_jobs=-1, verbose=-1, **kw)
    #if HAS_CAT: M["CatBoost"] = CatBoostClassifier(loss_function="Logloss" if is_binary else "MultiClass",depth=6, learning_rate=0.03, iterations=500, l2_leaf_reg=3,random_seed=SEED, verbose=0, thread_count=-1, auto_class_weights="Balanced")
    return M

def wrap_model(name, model):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if name in NEEDS_MINMAX: steps.append(("scaler", MinMaxScaler()))
    elif name in NEEDS_SCALING: steps.append(("scaler", StandardScaler()))
    steps.append(("clf", model))
    return Pipeline(steps)

def compute_metrics(y_true, y_pred, y_proba, is_binary):
    m = {"accuracy": accuracy_score(y_true, y_pred),
         "balanced_acc": balanced_accuracy_score(y_true, y_pred),
         "mcc": matthews_corrcoef(y_true, y_pred)}
    if is_binary:
        m["recall"]=recall_score(y_true,y_pred,zero_division=0)
        m["precision"]=precision_score(y_true,y_pred,zero_division=0)
        m["f1"]=f1_score(y_true,y_pred,zero_division=0)
        if y_proba is not None:
            try: m["auc"]=roc_auc_score(y_true,y_proba); m["pr_auc"]=average_precision_score(y_true,y_proba)
            except Exception: m["auc"]=m["pr_auc"]=np.nan
    else:
        m["recall_macro"]=recall_score(y_true,y_pred,average="macro",zero_division=0)
        m["precision_macro"]=precision_score(y_true,y_pred,average="macro",zero_division=0)
        m["f1_macro"]=f1_score(y_true,y_pred,average="macro",zero_division=0)
        m["f1_weighted"]=f1_score(y_true,y_pred,average="weighted",zero_division=0)
        if y_proba is not None:
            try: m["auc_ovr"]=roc_auc_score(y_true,y_proba,multi_class="ovr",average="macro")
            except Exception: m["auc_ovr"]=np.nan
    return m

def get_proba(pipe, X, is_binary):
    clf = pipe.named_steps["clf"]
    if hasattr(clf, "predict_proba"):
        try:
            p = pipe.predict_proba(X)
            return p[:,1] if is_binary else p
        except Exception: return None
    return None

# ════════════════════════════════════════════════════════════
print("="*70)
print("STEP 3 — TRANSDUCTIVE vs INDUCTIVE GRAPH EVALUATION")
print("="*70)

META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
results = []

for graph_mode in GRAPH_MODES:
    fpath = FEATURE_FILES[graph_mode]
    if not os.path.exists(fpath):
        print(f"\n⚠ {graph_mode}: {fpath} not found — skipping")
        continue
    print(f"\n{'#'*70}\n# GRAPH MODE: {graph_mode.upper()}\n{'#'*70}")
    df = pd.read_csv(fpath, low_memory=False)
    ALL_FEATURES = [c for c in df.columns if c not in META and c not in LEAKY_GRAPH]
    folds = sorted(df["fold"].unique())
    print(f"  Rows: {len(df):,}  Features: {len(ALL_FEATURES)}  Folds: {folds}")

    for task in TASKS:
        target_col = "label_enc" if task=="binary" else "class_enc"
        n_classes = df[target_col].nunique()
        is_binary = (task=="binary")
        print(f"\n  {'='*60}\n  TASK: {task} ({n_classes} classes) | {graph_mode}\n  {'='*60}")

        for variant_name, drop_cols in VARIANTS.items():
            feats = [f for f in ALL_FEATURES if f not in drop_cols]
            print(f"\n    {variant_name}: {len(feats)} features")
            for model_name, base_model in make_models(n_classes).items():
                t0 = time.time(); fold_metrics = []
                try:
                    for tf in folds:
                        tr = df[df["fold"]!=tf]; te = df[df["fold"]==tf]
                        pipe = wrap_model(model_name, base_model)
                        pipe.fit(tr[feats].values, tr[target_col].values)
                        y_pred = pipe.predict(te[feats].values)
                        y_proba = get_proba(pipe, te[feats].values, is_binary)
                        fold_metrics.append(compute_metrics(
                            te[target_col].values, y_pred, y_proba, is_binary))
                    avg = {k: np.nanmean([fm[k] for fm in fold_metrics]) for k in fold_metrics[0]}
                    std = {k: np.nanstd([fm[k] for fm in fold_metrics]) for k in fold_metrics[0]}
                    elapsed = time.time()-t0
                    row = {"graph_mode": graph_mode, "task": task, "variant": variant_name,
                           "model": model_name, "n_features": len(feats),
                           "train_time_s": round(elapsed,1)}
                    for k,v in avg.items():
                        row[k]=round(v,4); row[f"{k}_std"]=round(std[k],4)
                    results.append(row)
                    key = "auc" if is_binary else "f1_macro"
                    print(f"      {model_name:<18s}: {key}={avg.get(key,0):.4f} "
                          f"MCC={avg['mcc']:.4f} ({elapsed:.0f}s)")
                except Exception as e:
                    print(f"      {model_name:<18s}: FAILED — {type(e).__name__}: {e}")

res_df = pd.DataFrame(results)
res_df.to_csv(OUT_METRICS, index=False)
print(f"\n[saved] {OUT_METRICS}")

# ════════════════════════════════════════════════════════════
# TRANSDUCTIVE vs INDUCTIVE GAP REPORT
# ════════════════════════════════════════════════════════════
if len(res_df) and res_df["graph_mode"].nunique() == 2:
    print(f"\n{'='*70}\nTRANSDUCTIVE vs INDUCTIVE GAP (the headline analysis)\n{'='*70}")
    for task in TASKS:
        key = "auc" if task=="binary" else "f1_macro"
        sub = res_df[res_df["task"]==task]
        if key not in sub.columns: continue
        piv = sub.pivot_table(index=["model","variant"], columns="graph_mode", values=key)
        if "transductive" in piv and "inductive" in piv:
            piv["gap"] = piv["transductive"] - piv["inductive"]
            print(f"\n  {task.upper()} ({key}) — gap = transductive − inductive:")
            print(piv.round(4).sort_values("gap", ascending=False).head(8).to_string())
            print(f"\n  Mean gap: {piv['gap'].mean():.4f}")
            print(f"  → Small gap = graph signal is genuine & deployable")
            print(f"  → Large gap = graph signal depended on observing full corpus")

print(f"""
{'='*70}
STEP 3 COMPLETE — both graph modes evaluated
{'='*70}
  Manuscript framing:
    Transductive = full infrastructure graph observed (upper bound)
    Inductive    = train-only graph, unseen test URLs (deployment)
    The gap quantifies reliance on observing the complete campaign
    infrastructure vs. generalising from historical data alone.
  Next → Step 4 SHAP on the best inductive model
{'='*70}
""")

STEP 3 — TRANSDUCTIVE vs INDUCTIVE GRAPH EVALUATION

######################################################################
# GRAPH MODE: TRANSDUCTIVE
######################################################################
  Rows: 1,239,308  Features: 132  Folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]

  TASK: binary (2 classes) | transductive

    A_all: 132 features
      HistGB            : auc=0.9999 MCC=0.9921 (457s)
      RandomForest      : auc=0.9997 MCC=0.9845 (463s)
      LogisticRegression: auc=0.9992 MCC=0.9608 (188s)
      ComplementNB      : auc=0.9837 MCC=0.7243 (105s)
      XGBoost           : auc=0.9999 MCC=0.9874 (230s)
      LightGBM          : auc=0.9999 MCC=0.9906 (230s)

    B_no_tr: 131 features
      HistGB            : auc=0.9997 MCC=0.9720 (493s)
      RandomForest      : auc=0.9992 MCC=0.9704 (518s)
      LogisticRegression: auc=0.9961 MCC=0.9190 (244s)
      ComplementNB      : auc=0.9425 MCC=0.5635 (105s)
      XGBoost           : 

In [ ]:
import os, time, json, warnings
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, f1_score, recall_score, precision_score,
    matthews_corrcoef, balanced_accuracy_score, accuracy_score,
    average_precision_score)

warnings.filterwarnings("ignore")
SEED = 42

# ── PATHS ─────────────────────────────────────────────────────
DATA_DIR = (
    "/content/drive/MyDrive/Colab Notebooks/Research/New_Dataset_May"
)
INPUT_FILE  = os.path.join(DATA_DIR, "features_full_final.csv")
METRICS_CSV = os.path.join(DATA_DIR, "step3_all_metrics_final.csv")  # append here
TABNET_IMP  = os.path.join(DATA_DIR, "tabnet_attention_importance.csv")

# ── Detect deep libs ──────────────────────────────────────────
HAS_TABNET = HAS_FTT = HAS_TORCH = False
try:
    import torch
    HAS_TORCH = True
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
except ImportError:
    DEVICE = "cpu"
try:
    from pytorch_tabnet.tab_model import TabNetClassifier
    HAS_TABNET = True
except ImportError:
    pass
try:
    from tab_transformer_pytorch import FTTransformer
    HAS_FTT = True
except ImportError:
    pass

print("=" * 70)
print("STEP 3B — DEEP TABULAR MODELS")
print("=" * 70)
print(f"  Torch: {HAS_TORCH}  Device: {DEVICE}")
print(f"  TabNet: {HAS_TABNET}  FT-Transformer: {HAS_FTT}")
if DEVICE == "cpu":
    print("  ⚠ No GPU — TabNet will be slow. Enable GPU runtime in Colab.")

# ════════════════════════════════════════════════════════════
# CONFIG — match Step 3
# ════════════════════════════════════════════════════════════
TASKS    = ["binary", "5class"]
VARIANTS = {
    "A_all":         [],
    "B_no_tr":       ["is_tr_domain"],
    "C_no_tr_https": ["is_tr_domain", "is_https"],
}
# For speed: deep models only on the honest variant by default.
# Expand to all variants once you confirm it runs.
RUN_VARIANTS = ["C_no_tr_https"]   # change to list(VARIANTS) for full grid

# ════════════════════════════════════════════════════════════
# METRICS (identical to Step 3)
# ════════════════════════════════════════════════════════════
def compute_metrics(y_true, y_pred, y_proba, is_binary):
    m = {}
    m["accuracy"]     = accuracy_score(y_true, y_pred)
    m["balanced_acc"] = balanced_accuracy_score(y_true, y_pred)
    m["mcc"]          = matthews_corrcoef(y_true, y_pred)
    if is_binary:
        m["recall"]    = recall_score(y_true, y_pred, zero_division=0)
        m["precision"] = precision_score(y_true, y_pred, zero_division=0)
        m["f1"]        = f1_score(y_true, y_pred, zero_division=0)
        if y_proba is not None:
            try:
                m["auc"]    = roc_auc_score(y_true, y_proba)
                m["pr_auc"] = average_precision_score(y_true, y_proba)
            except Exception:
                m["auc"] = m["pr_auc"] = np.nan
    else:
        m["recall_macro"]    = recall_score(y_true, y_pred, average="macro", zero_division=0)
        m["precision_macro"] = precision_score(y_true, y_pred, average="macro", zero_division=0)
        m["f1_macro"]        = f1_score(y_true, y_pred, average="macro", zero_division=0)
        m["f1_weighted"]     = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        if y_proba is not None:
            try:
                m["auc_ovr"] = roc_auc_score(
                    y_true, y_proba, multi_class="ovr", average="macro")
            except Exception:
                m["auc_ovr"] = np.nan
    return m


# ════════════════════════════════════════════════════════════
# TABNET — train one fold
# ════════════════════════════════════════════════════════════
def train_tabnet(X_tr, y_tr, X_te, y_te, n_classes, feat_names):
    """Returns (y_pred, y_proba, feature_importance)."""
    # Impute + scale (TabNet benefits from scaling)
    imp = SimpleImputer(strategy="median")
    sc  = StandardScaler()
    X_tr_s = sc.fit_transform(imp.fit_transform(X_tr))
    X_te_s = sc.transform(imp.transform(X_te))

    # Carve a validation split from train (10%) for early stopping
    n_val  = max(1, int(0.1 * len(X_tr_s)))
    rng    = np.random.RandomState(SEED)
    perm   = rng.permutation(len(X_tr_s))
    val_idx, fit_idx = perm[:n_val], perm[n_val:]

    clf = TabNetClassifier(
        n_d=32, n_a=32, n_steps=5, gamma=1.5,
        lambda_sparse=1e-4,
        optimizer_fn=torch.optim.Adam,
        optimizer_params=dict(lr=2e-2),
        mask_type="entmax",
        scheduler_params=dict(step_size=50, gamma=0.9),
        scheduler_fn=torch.optim.lr_scheduler.StepLR,
        seed=SEED, verbose=0, device_name=DEVICE,
    )
    clf.fit(
        X_tr_s[fit_idx], y_tr[fit_idx],
        eval_set=[(X_tr_s[val_idx], y_tr[val_idx])],
        eval_metric=["accuracy"],
        max_epochs=100, patience=15,
        batch_size=4096, virtual_batch_size=512,
    )
    y_pred  = clf.predict(X_te_s)
    proba   = clf.predict_proba(X_te_s)
    y_proba = proba[:, 1] if n_classes == 2 else proba
    feat_imp = clf.feature_importances_   # native attention importance
    return y_pred, y_proba, feat_imp


# ════════════════════════════════════════════════════════════
# MAIN
# ════════════════════════════════════════════════════════════
if not HAS_TABNET:
    print("\n⚠ pytorch-tabnet not installed.")
    print("  Run: !pip install pytorch-tabnet -q")
    raise SystemExit

print(f"\n[1] Loading features ...")
df = pd.read_csv(INPUT_FILE, low_memory=False)
META = {"url","source","tld","label","label_enc",
        "class_final","class_enc","fold","reg_domain"}
ALL_FEATURES = [c for c in df.columns if c not in META]
folds = sorted(df["fold"].unique())
print(f"    Rows: {len(df):,}  Features: {len(ALL_FEATURES)}  Folds: {folds}")

# Load existing Step 3 metrics to append
if os.path.exists(METRICS_CSV):
    existing = pd.read_csv(METRICS_CSV)
    print(f"    Loaded existing leaderboard: {len(existing)} rows")
else:
    existing = pd.DataFrame()

new_rows = []
tabnet_importances = []

for task in TASKS:
    target_col = "label_enc" if task == "binary" else "class_enc"
    n_classes  = df[target_col].nunique()
    is_binary  = (task == "binary")
    print(f"\n{'='*70}")
    print(f"TASK: {task}  ({n_classes} classes)")
    print(f"{'='*70}")

    for variant_name in RUN_VARIANTS:
        drop_cols = VARIANTS[variant_name]
        feats = [f for f in ALL_FEATURES if f not in drop_cols]
        print(f"\n  ── {variant_name}: {len(feats)} features ──")

        # ── TabNet ───────────────────────────────────────────
        t0 = time.time()
        fold_metrics = []
        fold_imps    = []
        try:
            for test_fold in folds:
                tr = df[df["fold"] != test_fold]
                te = df[df["fold"] == test_fold]
                X_tr, y_tr = tr[feats].values, tr[target_col].values.astype(int)
                X_te, y_te = te[feats].values, te[target_col].values.astype(int)

                y_pred, y_proba, feat_imp = train_tabnet(
                    X_tr, y_tr, X_te, y_te, n_classes, feats)
                fm = compute_metrics(y_te, y_pred, y_proba, is_binary)
                fold_metrics.append(fm)
                fold_imps.append(feat_imp)
                print(f"      fold {test_fold}: done")

            avg = {k: np.nanmean([fm[k] for fm in fold_metrics])
                   for k in fold_metrics[0]}
            std = {k: np.nanstd([fm[k] for fm in fold_metrics])
                   for k in fold_metrics[0]}
            elapsed = time.time() - t0

            row = {"task": task, "variant": variant_name,
                   "model": "TabNet", "n_features": len(feats),
                   "train_time_s": round(elapsed, 1)}
            for k, v in avg.items():
                row[k] = round(v, 4)
                row[f"{k}_std"] = round(std[k], 4)
            new_rows.append(row)

            key = "auc" if is_binary else "f1_macro"
            print(f"    TabNet: {key}={avg.get(key, 0):.4f}  "
                  f"MCC={avg['mcc']:.4f}  ({elapsed:.0f}s)")

            # Save mean attention importance
            mean_imp = np.mean(fold_imps, axis=0)
            imp_df = pd.DataFrame({
                "feature": feats,
                "tabnet_importance": mean_imp,
                "task": task, "variant": variant_name,
            }).sort_values("tabnet_importance", ascending=False)
            tabnet_importances.append(imp_df)
            print(f"    Top-5 TabNet attention features:")
            for _, r in imp_df.head(5).iterrows():
                print(f"      {r['feature']:<25s}: {r['tabnet_importance']:.4f}")

        except Exception as e:
            print(f"    TabNet FAILED: {type(e).__name__}: {e}")

# ════════════════════════════════════════════════════════════
# SAVE — append to unified leaderboard
# ════════════════════════════════════════════════════════════
if new_rows:
    new_df = pd.DataFrame(new_rows)
    combined = pd.concat([existing, new_df], ignore_index=True) \
               if len(existing) else new_df
    combined.to_csv(METRICS_CSV, index=False)
    print(f"\n[2] Appended {len(new_rows)} rows → {METRICS_CSV}")

if tabnet_importances:
    pd.concat(tabnet_importances, ignore_index=True).to_csv(
        TABNET_IMP, index=False)
    print(f"    TabNet attention importance → {TABNET_IMP}")

# ════════════════════════════════════════════════════════════
# UNIFIED LEADERBOARD — classical vs deep
# ════════════════════════════════════════════════════════════
print(f"\n{'='*70}")
print("UNIFIED LEADERBOARD (classical + deep)")
print(f"{'='*70}")
final = pd.read_csv(METRICS_CSV)
for task in TASKS:
    sub = final[final["task"] == task]
    sort_col = "auc" if (task == "binary" and "auc" in sub.columns) else "f1_macro"
    if sort_col in sub.columns and len(sub):
        print(f"\n  {task.upper()} — top 8 by {sort_col}:")
        cols = [c for c in ["model","variant",sort_col,"mcc"] if c in sub.columns]
        print(sub.nlargest(8, sort_col)[cols].to_string(index=False))

print(f"""
{'='*70}
STEP 3B COMPLETE
{'='*70}
  Deep tabular model: TabNet (attention-based)
  Native interpretability: {TABNET_IMP}

  Thesis comparison now available:
    Classical trees (Step 3) vs Deep tabular (Step 3B)
    → if trees win: report "tree ensembles remain superior
      on engineered URL features despite deep alternatives"
    → if TabNet wins: report attention mechanism advantage

  Cross-check: compare TabNet attention top features against
  SHAP top features (Step 4) — agreement validates explainability

  Next → Step 4: SHAP (TreeExplainer) on best classical model
{'='*70}
""")

STEP 3B — DEEP TABULAR MODELS
  Torch: True  Device: cuda
  TabNet: True  FT-Transformer: False

[1] Loading features ...
    Rows: 1,239,308  Features: 134  Folds: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4)]
    Loaded existing leaderboard: 72 rows

TASK: binary  (2 classes)

  ── C_no_tr_https: 132 features ──

Early stopping occurred at epoch 51 with best_epoch = 36 and best_val_0_accuracy = 0.99898
      fold 0: done
Stop training because you reached max_epochs = 100 with best_epoch = 97 and best_val_0_accuracy = 0.99926
      fold 1: done

Early stopping occurred at epoch 45 with best_epoch = 30 and best_val_0_accuracy = 0.99909
      fold 2: done
Stop training because you reached max_epochs = 100 with best_epoch = 86 and best_val_0_accuracy = 0.99918
      fold 3: done

Early stopping occurred at epoch 88 with best_epoch = 73 and best_val_0_accuracy = 0.99927
      fold 4: done
    TabNet: auc=0.9998  MCC=0.9820  (7776s)
    Top-5 TabNet attention features:
